In [1]:
import random

# ==========================================
# 1. ENTIDAD DEL ENTORNO
# ==========================================
class Environment:
    def __init__(self, locations=['A', 'B'], initial_dirt=None, initial_agent_loc='A'):
        """Entorno configurable (Criterio 7)"""
        self.locations = locations
        # Si no se especifica, se genera suciedad aleatoria
        if initial_dirt is None:
            self.dirt_status = {loc: random.choice(['Limpio', 'Sucio']) for loc in locations}
        else:
            self.dirt_status = initial_dirt.copy()
            
        self.agent_location = initial_agent_loc
        self.performance_score = 0 # (Criterio 3)

    def get_percept(self):
        """Genera la percepción para el agente (Sensores)"""
        return (self.agent_location, self.dirt_status[self.agent_location])

    def execute_action(self, action):
        """Lógica de transición correcta y cálculo de rendimiento (Criterio 2 y 3)"""
        if action == 'Aspirar':
            if self.dirt_status[self.agent_location] == 'Sucio':
                self.dirt_status[self.agent_location] = 'Limpio'
                self.performance_score += 10  # Recompensa por limpiar
        
        elif action == 'Derecha':
            idx = self.locations.index(self.agent_location)
            if idx < len(self.locations) - 1:
                self.agent_location = self.locations[idx + 1]
            self.performance_score -= 1  # Penalización por moverse
            
        elif action == 'Izquierda':
            idx = self.locations.index(self.agent_location)
            if idx > 0:
                self.agent_location = self.locations[idx - 1]
            self.performance_score -= 1  # Penalización por moverse
            
        elif action == 'NoOp':
            pass # No penaliza ni recompensa

    def is_clean(self):
        """Verifica si todo el mundo está limpio"""
        return all(status == 'Limpio' for status in self.dirt_status.values())

# ==========================================
# 2. ENTIDADES DE LOS AGENTES
# ==========================================
class Agent:
    def act(self, percept):
        raise NotImplementedError("Debe implementarse en las subclases")

class RandomAgent(Agent):
    """Agente que elige acciones al azar (Criterio 8)"""
    def act(self, percept):
        return random.choice(['Izquierda', 'Derecha', 'Aspirar', 'NoOp'])

class SimpleReflexAgent(Agent):
    """Agente Reactivo Simple basado en reglas (Criterio 8)"""
    def act(self, percept):
        location, status = percept
        
        if status == 'Sucio':
            return 'Aspirar'
        elif location == 'A':
            return 'Derecha'
        elif location == 'B':
            return 'Izquierda'
        else:
            return 'NoOp'

# ==========================================
# 3. EVALUADOR / SIMULADOR
# ==========================================
def run_simulation(agent, environment, steps=10):
    """Ejecuta la simulación separando estrictamente Agente y Entorno (Criterios 5 y 6)"""
    for _ in range(steps):
        if environment.is_clean():
            break # Termina anticipadamente si ya limpió todo
            
        percept = environment.get_percept()
        action = agent.act(percept)
        environment.execute_action(action)
        
    return environment.performance_score

def run_experiments(num_trials=10):
    """Ejecuta múltiples ensayos y compara (Criterio 9)"""
    score_random = 0
    score_reflex = 0
    
    for _ in range(num_trials):
        # Configuraciones iniciales aleatorias idénticas para una comparación justa
        initial_dirt = {'A': random.choice(['Limpio', 'Sucio']), 
                        'B': random.choice(['Limpio', 'Sucio'])}
        start_loc = random.choice(['A', 'B'])
        
        # Prueba Agente Aleatorio
        env_rand = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_rand = RandomAgent()
        score_random += run_simulation(agent_rand, env_rand)
        
        # Prueba Agente Reactivo Simple
        env_refl = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_refl = SimpleReflexAgent()
        score_reflex += run_simulation(agent_refl, env_refl)
        
    print(f"--- Resultados tras {num_trials} ensayos ---")
    print(f"Puntuación Promedio Agente Aleatorio: {score_random / num_trials}")
    print(f"Puntuación Promedio Agente Reactivo: {score_reflex / num_trials}")

# Ejecutar el experimento
if __name__ == "__main__":
    run_experiments(100)

--- Resultados tras 100 ensayos ---
Puntuación Promedio Agente Aleatorio: 2.92
Puntuación Promedio Agente Reactivo: 8.45
